In [5]:
import json
import os
import re
import time

from openai import OpenAI
from pymilvus import DataType, MilvusClient
from pymilvus import model as milvus_model


# ===================== 配置 =====================
class Config:
    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    MILVUS_DB_PATH = "products.db"
    COLLECTION_NAME = "skincare_products"
    MODEL_NAME = "deepseek-chat"
    BASE_URL = "https://api.deepseek.com/v1"


# ===================== Milvus 向量数据库封装 =====================
class ProductVectorDB:
    def __init__(self):
        self.embedding_model = milvus_model.DefaultEmbeddingFunction()
        self.embedding_dim = len(self.embedding_model.encode_queries(["test"])[0])

        self.client = MilvusClient(Config.MILVUS_DB_PATH)
        self.collection_name = Config.COLLECTION_NAME
        self._init_collection()

    def _init_collection(self):
        if self.collection_name in self.client.list_collections():
            self.client.drop_collection(self.collection_name)

        schema = self.client.create_schema(
            auto_id=False,
            enable_dynamic_fields=False,
        )
        schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
        schema.add_field(field_name="name", datatype=DataType.VARCHAR, max_length=200)
        schema.add_field(field_name="info", datatype=DataType.VARCHAR, max_length=2000)
        schema.add_field(
            field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=self.embedding_dim
        )

        index_params = self.client.prepare_index_params()
        index_params.add_index(
            field_name="vector", index_type="AUTOINDEX", metric_type="COSINE"
        )

        self.client.create_collection(
            collection_name=self.collection_name,
            schema=schema,
            index_params=index_params,
        )

    def encode(self, text: str):
        return self.embedding_model.encode_queries([text])[0]

    def insert_products(self, products):
        data = []
        for p in products:
            data.append(
                {
                    "id": p["id"],
                    "name": p["name"],
                    "info": p["info"],
                    "vector": self.encode(p["name"] + " " + p["info"]),
                }
            )
        self.client.insert(collection_name=self.collection_name, data=data)
        print(f"✅ 已插入 {len(products)} 个产品到 Milvus 向量库")

    def search_product(self, product_name: str):
        res = self.client.search(
            collection_name=self.collection_name,
            data=[self.encode(product_name)],
            filter=f'name like "%{product_name}%"',
            limit=1,
            output_fields=["name", "info"],
        )
        if res and res[0]:
            return res[0][0]["entity"]["info"]
        return "未找到产品信息"


# ===================== 工具类 =====================
class ProductTools:
    def __init__(self, vector_db: ProductVectorDB):
        self.vector_db = vector_db

    def search_web(self, query: str):
        time.sleep(0.5)
        return "小红书美妆趋势：补水保湿、修护屏障、抗老、敏感肌、水光肌、熬夜急救"

    def query_product_database(self, product_name: str):
        print(f"[Tool Call] 从 Milvus 查询产品：{product_name}")
        time.sleep(0.7)
        return self.vector_db.search_product(product_name)

    def generate_emoji(self, context: str):
        time.sleep(0.2)
        if "补水" in context:
            return ["💦", "💧", "✨", "🌊"]
        return ["✨", "🔥", "💖", "💯", "🎉"]


# ===================== AI Agent =====================
class XiaohongshuAgent:
    def __init__(self, tools: ProductTools):
        self.client = OpenAI(api_key=Config.DEEPSEEK_API_KEY, base_url=Config.BASE_URL)
        self.tools = tools
        self.available_tools = {
            "search_web": self.tools.search_web,
            "query_product_database": self.tools.query_product_database,
            "generate_emoji": self.tools.generate_emoji,
        }

    def generate(self, product_name, tone="活泼甜美", max_iter=5):
        messages = [
            {
                "role": "system",
                "content": """你是资深小红书爆款文案专家，生成活泼、真诚、高互动文案。
最终输出严格JSON格式：
{
  "title": "标题",
  "body": "正文",
  "hashtags": ["标签1","标签2","标签3"],
  "emojis": ["✨","🔥"]
}""",
            },
            {
                "role": "user",
                "content": f"请为产品「{product_name}」生成小红书文案，风格{tone}",
            },
        ]

        # ========== 这里全部修复缩进！==========
        for _ in range(max_iter):
            response = self.client.chat.completions.create(
                model=Config.MODEL_NAME,
                messages=messages,
                tools=[
                    {
                        "type": "function",
                        "function": {
                            "name": "search_web",
                            "description": "Search for latest beauty trends",
                            "parameters": {
                                "type": "object",
                                "properties": {"query": {"type": "string"}},
                                "required": ["query"],
                            },
                        },
                    },
                    {
                        "type": "function",
                        "function": {
                            "name": "query_product_database",
                            "description": "Query product information",
                            "parameters": {
                                "type": "object",
                                "properties": {"product_name": {"type": "string"}},
                                "required": ["product_name"],
                            },
                        },
                    },
                    {
                        "type": "function",
                        "function": {
                            "name": "generate_emoji",
                            "description": "Generate emojis based on context",
                            "parameters": {
                                "type": "object",
                                "properties": {"context": {"type": "string"}},
                                "required": ["context"],
                            },
                        },
                    },
                ],
                tool_choice="auto",
            )

            msg = response.choices[0].message
            if msg.tool_calls:
                messages.append(msg)
                for tool in msg.tool_calls:
                    func = self.available_tools[tool.function.name]
                    args = json.loads(tool.function.arguments or "{}")
                    res = func(**args)
                    messages.append(
                        {"tool_call_id": tool.id, "role": "tool", "content": str(res)}
                    )
            else:
                match = re.search(r"```json\s*(.*?)\s*```", msg.content, re.DOTALL)
                try:
                    data = json.loads(match.group(1) if match else msg.content)
                    return json.dumps(data, ensure_ascii=False, indent=2)
                except:
                    continue
        return "生成失败"


# ===================== 10个新产品 =====================
SAMPLE_PRODUCTS = [
    {
        "id": 1,
        "name": "深海蓝藻保湿面膜",
        "info": "蓝藻提取物，深层补水、修护屏障、敏感肌可用",
    },
    {"id": 2, "name": "玻尿酸水光精华", "info": "高浓度玻尿酸，快速补水、提亮肤色"},
    {"id": 3, "name": "烟酰胺美白乳液", "info": "5%烟酰胺，淡化痘印、均匀肤色"},
    {"id": 4, "name": "神经酰胺修护面霜", "info": "修护屏障，舒缓泛红，干敏皮救星"},
    {"id": 5, "name": "茶树控油洁面乳", "info": "深层清洁，控油祛痘，温和不紧绷"},
    {"id": 6, "name": "维A醇抗老精华", "info": "淡化细纹，紧致肌肤，提升弹性"},
    {"id": 7, "name": "金盏花舒缓爽肤水", "info": "舒缓镇静，收缩毛孔，补水保湿"},
    {"id": 8, "name": "维生素C亮肤原液", "info": "抗氧化，提亮肤色，改善暗沉"},
    {"id": 9, "name": "氨基酸泡沫洗面奶", "info": "温和清洁，保湿不紧绷，适合所有肤质"},
    {"id": 10, "name": "胶原蛋白补水面膜", "info": "补充胶原，紧致肌肤，长效保湿"},
]

# ===================== 运行 =====================
if __name__ == "__main__":
    db = ProductVectorDB()
    db.insert_products(SAMPLE_PRODUCTS)

    tools = ProductTools(db)
    agent = XiaohongshuAgent(tools)

    result = agent.generate("深海蓝藻保湿面膜")
    print("\n===== 最终文案 =====")
    print(result)

/home/ericz/miniconda3/envs/deepseek_env/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ 已插入 10 个产品到 Milvus 向量库
[Tool Call] 从 Milvus 查询产品：深海蓝藻保湿面膜

===== 最终文案 =====
{
  "title": "脸蛋就像喝饱了深海矿泉水💧这面膜也太水嫩了吧！",
  "body": "姐妹们！我挖到宝了😭❤️\n\n最近换季皮肤干到起皮\n上妆卡粉卡到怀疑人生🥲\n直到我遇到了这罐「深海蓝藻保湿面膜」！！！\n\n🌊 第一次敷就被惊艳到！\n像把一整片大海拍在脸上的感觉🌊\n蓝藻提取物+海洋深层水\n补水真的不是说说而已~\n\n🍃质地是那种冰冰凉凉的啫喱状\n敷上去超舒服 像在给皮肤做SPA💆‍♀️\n敷完15分钟揭下来\n脸蛋摸起来QQ弹弹的 像剥了壳的鸡蛋🥚✨\n\n💙最绝的是！\n第二天起床皮肤还是水嘟嘟的！\n一点都不夸张 就像皮肤喝饱了水！\n上妆再也不卡粉啦~~\n而且敏感肌用着也完全ok！\n\n姐妹们冲就完事了！\n这个价格能买到这种补水效果\n我真的会谢！！！🙏💕\n\n#平价补水面膜 #深海蓝藻 #拯救沙漠大干皮 #换季护肤 #敏感肌也能用 #补水面膜推荐 #护肤日常 #好物分享"
}
